In [0]:
%pip install databricks-feature-engineering codecarbon tensorflow

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn import datasets
import pyspark
import re
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from sklearn.preprocessing import StandardScaler
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

csv_path = "/Volumes/workspace/athletes/athletes/athletes.csv"

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)   
    .option("escape", '"')       
    .csv(csv_path)
)

display(raw_df.limit(10))
print(raw_df.count())
raw_df.printSchema()


In [0]:
raw_df.write.mode("overwrite").saveAsTable("workspace.athletes.athletes_raw")

In [0]:
df = spark.table("workspace.athletes.athletes_raw")

In [0]:
fe = FeatureEngineeringClient()
CATALOG_SCHEMA = "workspace.athletes"

REQUIRED_COLS = ['athlete_id', 'region', 'age', 'weight', 'height', 'gender',
                 'eat', 'train', 'background', 'experience', 'schedule', 'howlong',
                 'deadlift', 'candj', 'snatch', 'backsq']
NUMERIC_COLS = ['age', 'weight', 'height', 'deadlift', 'candj', 'snatch', 'backsq']
LIFT_COLS = ['deadlift', 'candj', 'snatch', 'backsq']
SURVEY_COLS = ['background', 'experience', 'schedule', 'howlong', 'eat']
MULTISELECT_COLS = ['eat', 'background', 'schedule', 'experience']


def _sanitize_columns(cols):
    """Delta column names can't contain spaces, commas, parens, etc."""
    seen, out = {}, []
    for c in cols:
        new = re.sub(r'[^0-9a-zA-Z]+', '_', c).strip('_').lower()
        if new in seen:
            seen[new] += 1
            new = f"{new}_{seen[new]}"
        else:
            seen[new] = 0
        out.append(new)
    return out


def build_and_write_features(raw_sdf, version):
    """
    v1: keeps athletes who declined survey questions (declines -> all-zero dummies)
    v2: drops athletes who declined any survey question
    Writes features to the feature store and labels to a separate table.
    """
    assert version in ("v1", "v2")

    # Only pull needed columns into pandas
    df = raw_sdf.select(*REQUIRED_COLS).toPandas()
    for c in NUMERIC_COLS:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # Remove missing rows + not relevant columns
    df = df.dropna(subset=REQUIRED_COLS).drop(columns=['train'])

    # Remove outliers
    df = df[df['weight'] < 1500]
    df = df[df['gender'] != '--']
    df = df[df['age'] >= 18]
    df = df[(df['height'] < 96) & (df['height'] > 48)]

    male_ok = (df['gender'] == 'Male') & (df['deadlift'] <= 1105)
    female_ok = (df['gender'] == 'Female') & (df['deadlift'] <= 636)
    df = df[(df['deadlift'] > 0) & (male_ok | female_ok)]

    df = df[(df['candj'] > 0) & (df['candj'] <= 395)]
    df = df[(df['snatch'] > 0) & (df['snatch'] <= 496)]
    df = df[(df['backsq'] > 0) & (df['backsq'] <= 1069)]

    # Clean survey data
    df = df.replace({'Decline to answer|': np.nan})
    if version == "v2":
        df = df.dropna(subset=SURVEY_COLS)

    # Primary key must be unique + non-null
    df['athlete_id'] = df['athlete_id'].astype('int64')
    df = df.drop_duplicates(subset='athlete_id')

    # Labels go in their own table (lifts are the target -> leakage if kept as features)
    labels = df[['athlete_id']].assign(total_lift=df[LIFT_COLS].sum(axis=1))
    feats = df.drop(columns=LIFT_COLS)

    # One-hot encode multiselect survey columns
    for col in MULTISELECT_COLS:
        dummies = feats[col].str.get_dummies(sep='|').add_prefix(f'{col}_')
        feats = pd.concat([feats.drop(columns=col), dummies], axis=1)
    feats = feats.drop(columns=feats.filter(like='Decline to answer').columns)

    feats = pd.get_dummies(feats, columns=['region', 'gender', 'howlong'],
                           drop_first=True, dtype=int)
    feats.columns = _sanitize_columns(feats.columns)

    # Write features to feature store
    feature_table = f"{CATALOG_SCHEMA}.athlete_features_{version}"
    feats_sdf = spark.createDataFrame(feats)

    if spark.catalog.tableExists(feature_table):
        fe.write_table(name=feature_table, df=feats_sdf, mode="merge")
    else:
        fe.create_table(
            name=feature_table,
            primary_keys=["athlete_id"],
            df=feats_sdf,
            description=f"Athlete features {version}",
        )

    # Write labels
    label_table = f"{CATALOG_SCHEMA}.athlete_labels_{version}"
    spark.createDataFrame(labels).write.mode("overwrite").saveAsTable(label_table)

    print(f"{version}: {len(feats):,} rows, {feats.shape[1] - 1} features -> {feature_table}")
    return feature_table, label_table


raw = spark.table("workspace.athletes.athletes_raw")
v1_feature_table, v1_label_table = build_and_write_features(raw, "v1")
v2_feature_table, v2_label_table = build_and_write_features(raw, "v2")

In [0]:
def prepare_data(version, test_pct=20, seed=30):
    """
    Pull features from the feature store, split train/test, and scale.
    Split is deterministic per athlete_id, so the same athlete always lands
    in the same split across versions and runs.
    """
    feature_table = f"{CATALOG_SCHEMA}.athlete_features_{version}"
    label_table = f"{CATALOG_SCHEMA}.athlete_labels_{version}"

    # Train/test split on athlete_id (before feature lookup)
    labels = spark.table(label_table)
    bucket = F.pmod(F.xxhash64(F.col("athlete_id"), F.lit(seed)), F.lit(100))
    train_labels = labels.filter(bucket >= test_pct)
    test_labels = labels.filter(bucket < test_pct)

    # Feature store lookup
    lookups = [FeatureLookup(table_name=feature_table, lookup_key="athlete_id")]

    def _make_set(label_df):
        return fe.create_training_set(
            df=label_df,
            feature_lookups=lookups,
            label="total_lift",
            exclude_columns=["athlete_id"],
        )

    train_set = _make_set(train_labels)
    test_set = _make_set(test_labels)

    train_pdf = train_set.load_df().toPandas()
    test_pdf = test_set.load_df().toPandas()

    X_train = train_pdf.drop(columns=["total_lift"])
    y_train = train_pdf["total_lift"]
    X_test = test_pdf.drop(columns=["total_lift"])[X_train.columns]  # same column order
    y_test = test_pdf["total_lift"]

    # Scaling (fit on train only)
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),
                                  columns=X_train.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test),
                                 columns=X_test.columns, index=X_test.index)

    print(f"{version}: train={len(X_train):,}  test={len(X_test):,}  features={X_train.shape[1]}")

    return {
        "training_set": train_set,   # pass to fe.log_model for feature lineage
        "scaler": scaler,
        "X_train": X_train, "X_test": X_test,
        "X_train_scaled": X_train_scaled, "X_test_scaled": X_test_scaled,
        "y_train": y_train, "y_test": y_test,
    }


data = {v: prepare_data(v) for v in ["v1", "v2"]}

In [0]:
import mlflow
import tensorflow as tf
from codecarbon import OfflineEmissionsTracker
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mlflow.autolog(disable=True)  # manual logging only, avoids duplicate runs
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/athletes_total_lift")

HYPERPARAMS = {
    "hp_a": {"learning_rate": 0.05, "epochs": 20, "batch_size": 250, "units": [64, 32],  "dropout": 0.0},
    "hp_b": {"learning_rate": 0.01, "epochs": 40, "batch_size": 128, "units": [128, 64], "dropout": 0.2},
}


def run_experiment(version, hp_name, hp):
    d = data[version]  # from prepare_data()
    tf.keras.utils.set_random_seed(30)

    X_tr = d["X_train_scaled"].astype("float32")
    X_te = d["X_test_scaled"].astype("float32")
    y_scaler = StandardScaler()
    y_tr = y_scaler.fit_transform(d["y_train"].to_numpy().reshape(-1, 1)).astype("float32")
    y_te = d["y_test"].to_numpy()

    # Same MLP architecture family, sized by hyperparameters
    layers = [tf.keras.layers.Input(shape=(X_tr.shape[1],))]
    for units in hp["units"]:
        layers += [tf.keras.layers.Dense(units, activation="relu"),
                   tf.keras.layers.Dropout(hp["dropout"])]
    model = tf.keras.Sequential(layers + [tf.keras.layers.Dense(1)])
    model.compile(optimizer=tf.keras.optimizers.SGD(hp["learning_rate"]), loss="mse")

    with mlflow.start_run(run_name=f"{version}_{hp_name}"):
        mlflow.set_tags({"feature_version": version, "hp_set": hp_name})
        mlflow.log_params({
            **hp, "units": str(hp["units"]),
            "feature_table": f"workspace.athletes.athlete_features_{version}",
            "n_features": X_tr.shape[1], "n_train": len(X_tr),
        })

        # Track emissions for training only
        tracker = OfflineEmissionsTracker(country_iso_code="USA", measure_power_secs=1,
                                          save_to_file=False, log_level="error")
        tracker.start()
        history = model.fit(X_tr, y_tr, epochs=hp["epochs"], batch_size=hp["batch_size"],
                            validation_split=0.1, verbose=0)
        emissions = tracker.stop()

        pred = y_scaler.inverse_transform(model.predict(X_te, verbose=0)).ravel()
        metrics = {
            "test_mae": mean_absolute_error(y_te, pred),
            "test_rmse": mean_squared_error(y_te, pred) ** 0.5,
            "test_r2": r2_score(y_te, pred),
            "emissions_kg": emissions,
            "energy_kwh": tracker.final_emissions_data.energy_consumed,
            "train_seconds": tracker.final_emissions_data.duration,
        }
        mlflow.log_metrics(metrics)

        # Qualitative plots
        fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
        ax[0].plot(history.history["loss"], label="train")
        ax[0].plot(history.history["val_loss"], label="val")
        ax[0].set(title="Loss (scaled MSE)", xlabel="epoch"); ax[0].legend()

        lims = [min(y_te.min(), pred.min()), max(y_te.max(), pred.max())]
        ax[1].scatter(y_te, pred, s=4, alpha=0.3)
        ax[1].plot(lims, lims, "r--")
        ax[1].set(title="Predicted vs actual", xlabel="actual total_lift", ylabel="predicted")

        ax[2].hist(y_te - pred, bins=50)
        ax[2].set(title="Residuals", xlabel="actual - predicted")

        fig.suptitle(f"{version} / {hp_name}")
        fig.tight_layout()
        mlflow.log_figure(fig, "diagnostics.png")
        plt.close(fig)

    print(f"{version}/{hp_name}: RMSE={metrics['test_rmse']:.1f}  R2={metrics['test_r2']:.3f}  CO2={emissions:.2e} kg")


# ---- 4 experiments: 2 feature versions x 2 hyperparameter sets ----
for version in ["v1", "v2"]:
    for hp_name, hp in HYPERPARAMS.items():
        run_experiment(version, hp_name, hp)

# ---- Comparison table (quantitative + emissions) ----
runs = mlflow.search_runs(order_by=["metrics.test_rmse ASC"])
cols = ["tags.mlflow.runName", "tags.feature_version", "tags.hp_set",
        "metrics.test_rmse", "metrics.test_mae", "metrics.test_r2",
        "metrics.emissions_kg", "metrics.energy_kwh", "metrics.train_seconds"]
display(runs[cols])

In [0]:
import os, shutil

repo_results = "/Workspace/Users/oliphantj@uchicago.edu/dbx_hw2/results"
os.makedirs(repo_results, exist_ok=True)

runs = mlflow.search_runs(order_by=["attributes.start_time DESC"], max_results=4)

for _, r in runs.iterrows():
    name = r["tags.mlflow.runName"]
    local = mlflow.artifacts.download_artifacts(run_id=r["run_id"], artifact_path="diagnostics.png")
    shutil.copy(local, f"{repo_results}/{name}.png")

# Save the comparison table too
cols = ["tags.mlflow.runName", "tags.feature_version", "tags.hp_set",
        "metrics.test_rmse", "metrics.test_mae", "metrics.test_r2",
        "metrics.emissions_kg", "metrics.energy_kwh", "metrics.train_seconds"]
runs[cols].to_csv(f"{repo_results}/experiment_comparison.csv", index=False)

print(os.listdir(repo_results))